# Chốt CQRS — đo bản cuối F so với O: `cqrs-final-matched-kaggle-v1`

Kaggle **CPU / Internet On**. Source lấy từ Kaggle Dataset `cqrs-list-source-6b510cb9.bin` (Add Input), kiểm SHA256 cố định: O (`3e88d7e`, trước CQRS) và F (bản cuối: danh sách đọc nguồn, `CLINIC_READ_PROJECTION=off` mặc định, không projector, không read diagnostics). Cùng workload, generator, envelope và cách chấm validity như list study v3; chỉ có hai cấu hình và cổng là **F/O không regression** (ngưỡng lấy từ v3: p95 ≤ 1,1×, CPU/success ≤ 1,1×, throughput ≥ 0,9×).

**Chọn chế độ ở cell dưới:**
- `MODE='smoke'` (mặc định, khoảng 1 giờ): readiness. Gồm calibration 120s và một block O,F ngắn cho mỗi workload. Không phải phép đo.
- `MODE='full'`: readiness smoke của workload đó, **rồi** các block đã chọn. Đề xuất chia phiên:
  - phiên 1: `WORKLOADS=['query-heavy']`, `BLOCKS=[0,1,2]` (khoảng 3–3,5 giờ);
  - phiên 2: `WORKLOADS=['pcn']`, `BLOCKS=[0,1,2]` (khoảng 4,5–5 giờ).

  Chạy full bằng **Save Version → Save & Run All (Commit)** để không bị ngắt vì idle.

Không retry, không giảm tải, không đổi quota. Raw được giữ cả khi FAIL. Kết luận chỉ có sau khi gộp đủ 3 block cùng profileKey của mỗi workload.

In [ ]:
MODE = 'smoke'            # 'smoke' | 'full'
WORKLOADS = ['query-heavy', 'pcn']
BLOCKS = [0, 1, 2]        # only used when MODE == 'full'
assert MODE in ('smoke', 'full') and WORKLOADS and set(WORKLOADS) <= {'query-heavy', 'pcn'} and len(set(WORKLOADS)) == len(WORKLOADS)
assert set(BLOCKS) <= {0, 1, 2} and len(set(BLOCKS)) == len(BLOCKS)
import base64, hashlib, importlib.util, io, json, os, platform, sys, uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir(), 'Chạy trong notebook Kaggle.'
assert os.geteuid() == 0, 'Setup native cần quyền root của kernel.'

SOURCE_SHA256 = '6b510cb95e9931c45ef3bae51dc0481733e6249f2c7aa09d92c004b4576dc56c'
# The source archive comes from an attached private Kaggle Dataset (Add Input);
# the notebook pins its SHA256, so any other file is refused before use.
ARCHIVE_NAME = 'cqrs-list-source-6b510cb9.bin'
candidates = sorted(Path('/kaggle/input').rglob(ARCHIVE_NAME))
assert len(candidates) == 1, 'Attach the Kaggle Dataset containing ' + ARCHIVE_NAME + ' (Add Input); found: ' + str(len(candidates))
raw = candidates[0].read_bytes()
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, 'Source archive hash mismatch'
with ZipFile(io.BytesIO(raw)) as z:
    validator = z.read('source/benchmarks/phase2/kaggle/make_final_notebook.py')
namespace = {'__name__': 'list_bundle_validator', '__file__': '/tmp/list-validator/benchmarks/phase2/kaggle/make_final_notebook.py'}
exec(compile(validator, '<verified-list-validator>', 'exec'), namespace)
source_manifest, values = namespace['verified_members'](raw, SOURCE_SHA256)
RUN = str(uuid.uuid4())
PRIVATE = Path('/tmp') / ('cqrs-final-' + RUN); OUTPUT = Path('/kaggle/working') / ('cqrs-final-' + RUN)
PRIVATE.mkdir(mode=0o700); OUTPUT.mkdir(mode=0o700)
ROOT = PRIVATE / 'source'; ROOT.mkdir(mode=0o700)
for name, value in values.items():
    destination = ROOT.joinpath(*PurePosixPath(name).parts)
    assert destination.resolve().is_relative_to(ROOT.resolve())
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('xb') as handle:
        handle.write(value)
sys.path.insert(0, str(ROOT / 'benchmarks/phase2/kaggle'))
spec = importlib.util.spec_from_file_location('final_study', ROOT / 'benchmarks/phase2/kaggle/final_study.py')
study = importlib.util.module_from_spec(spec); spec.loader.exec_module(study)
print({'run': RUN, 'study': study.base.STUDY, 'source_verified': True, 'source_archive_sha256': SOURCE_SHA256, 'mode': MODE, 'workloads': WORKLOADS})

## Chạy
In tiến độ mỗi 30s. Khi lỗi vẫn export ZIP ghi rõ stage lỗi. Log, context và token chỉ nằm trong `/tmp`, không đưa ra ngoài.

In [ ]:
summary = study.run_study(ROOT, PRIVATE, OUTPUT, RUN, mode=MODE, workloads=WORKLOADS, blocks=BLOCKS if MODE == 'full' else None)
print(json.dumps({k: v for k, v in summary.items() if k != 'download'}, indent=2))
from IPython.display import display, FileLink
display(FileLink(summary['download']))

## Gửi kết quả
Tải file `.zip` ở trên, hoặc lấy ở tab Output của bản commit, rồi gửi lại để review. Mỗi phiên chỉ là một phần; chỉ gộp các block khi chúng cùng profileKey.